In [1]:
%pip install -q langchain langgraph langchain-openai langchain-huggingface \
    pydantic python-dotenv gradio

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from typing import TypedDict

from langgraph.graph import StateGraph, START, END

In [3]:
class FoodState(TypedDict):
    user_request: str
    people: int | None
    budget: float | None
    preference: str | None
    recommended_food: str | None
    ingredients: list[str]
    restaurant: str | None
    price_per_person: float | None
    total_cost: float | None
    remaining_budget: float | None
    budget_status: str | None
    final_response: str | None

In [4]:

def analyze_request(state: FoodState):
    return {
        "people": state["people"],
        "budget": state["budget"],
        "preference": state["preference"]
    }
workflow = StateGraph(FoodState)

workflow.add_node("analyze_request", analyze_request)

workflow.add_edge(START, "analyze_request")
workflow.add_edge("analyze_request", END)

app = workflow.compile()

In [5]:
initial_state = {
    "user_request": "I want Egyptian food for 2 people with a budget of 300 EGP",

    "people": 2,
    "budget": 300,
    "preference": "traditional Egyptian",

    "recommended_food": None,

    "ingredients": [],

    "restaurant": None,
    "price_per_person": None,

    "total_cost": None,
    "remaining_budget": None,

    "budget_status": None,

    "final_response": None
}

result = app.invoke(initial_state)

result

{'user_request': 'I want Egyptian food for 2 people with a budget of 300 EGP',
 'people': 2,
 'budget': 300,
 'preference': 'traditional Egyptian',
 'recommended_food': None,
 'ingredients': [],
 'restaurant': None,
 'price_per_person': None,
 'total_cost': None,
 'remaining_budget': None,
 'budget_status': None,
 'final_response': None}

In [6]:
def food_agent(state: FoodState):

    preference = state["preference"]

    if "traditional" in preference.lower():
        food = "Koshary"
    else:
        food = "Koshary"

    return {
        "recommended_food": food
    }
workflow = StateGraph(FoodState)

workflow.add_node("analyze_request", analyze_request)
workflow.add_node("food_agent", food_agent)

workflow.add_edge(START, "analyze_request")
workflow.add_edge("analyze_request", "food_agent")
workflow.add_edge("food_agent", END)

app = workflow.compile()

In [7]:
result = app.invoke(initial_state)

print(result["recommended_food"])

Koshary


In [8]:
FOOD_DATA = {
    "Koshary": [
        "Rice",
        "Lentils",
        "Macaroni",
        "Tomato sauce",
        "Chickpeas",
        "Fried onions"
    ],

    "Foul": [
        "Fava beans",
        "Garlic",
        "Lemon",
        "Cumin",
        "Olive oil"
    ],

    "Taamia": [
        "Fava beans",
        "Parsley",
        "Coriander",
        "Garlic",
        "Cumin"
    ]
}
def ingredients_agent(state: FoodState):

    food = state["recommended_food"]

    ingredients = FOOD_DATA.get(food, [])

    return {
        "ingredients": ingredients
    }

In [9]:
RESTAURANTS = {
    "Koshary": [
        {
            "name": "Koshary Place A",
            "price": 50
        },
        {
            "name": "Koshary Place B",
            "price": 60
        }
    ],

    "Foul": [
        {
            "name": "Foul Place A",
            "price": 35
        }
    ]
}
def restaurant_agent(state: FoodState):

    food = state["recommended_food"]

    options = RESTAURANTS.get(food, [])

    if not options:
        return {
            "restaurant": None,
            "price_per_person": None
        }

    best_option = options[0]

    return {
        "restaurant": best_option["name"],
        "price_per_person": best_option["price"]
    }

In [10]:
def bill_agent(state: FoodState):

    people = state["people"]
    price = state["price_per_person"]
    budget = state["budget"]

    total = people * price
    remaining = budget - total

    return {
        "total_cost": total,
        "remaining_budget": remaining
    }

                  Bill Agent
                      ↓
                 Check Budget
                  /         \
                 /           \
              OK             Too expensive
              ↓                   ↓
         Final Agent          Food Agent
                                  ↓
                             Restaurant
                                  ↓
                                Bill

In [11]:
def check_budget(state: FoodState):

    if state["total_cost"] <= state["budget"]:
        return "final_agent"

    return "food_agent"

                       START
                         │
                         ▼
                 Analyze Request
                         │
                         ▼
                    Food Agent
                         │
                         ▼
                 Ingredients Agent
                         │
                         ▼
                 Restaurant Agent
                         │
                         ▼
                    Bill Agent
                         │
                         ▼
                  Budget Checker
                    /         \
                   /           \
                  ▼             ▼
           Final Agent       Food Agent
                  │              │
                  ▼              │
                 END       Restaurant Agent
                                 │
                                 ▼
                              Bill Agent

In [12]:
def final_agent(state: FoodState):
    return {
        "final_response": (
            f"Recommended food: {state['recommended_food']}\n"
            f"Restaurant: {state['restaurant']}\n"
            f"People: {state['people']}\n"
            f"Total cost: {state['total_cost']} EGP\n"
            f"Remaining budget: {state['remaining_budget']} EGP"
        )
    }

In [13]:
workflow = StateGraph(FoodState)

workflow.add_node("analyze_request", analyze_request)
workflow.add_node("food_agent", food_agent)
workflow.add_node("ingredients_agent", ingredients_agent)
workflow.add_node("restaurant_agent", restaurant_agent)
workflow.add_node("calculate_bill", bill_agent)
workflow.add_node("final_agent", final_agent)

In [14]:
workflow.add_edge(START, "analyze_request")

workflow.add_edge("analyze_request", "food_agent")

workflow.add_edge("food_agent", "ingredients_agent")

workflow.add_edge("ingredients_agent", "restaurant_agent")

workflow.add_edge("restaurant_agent", "calculate_bill")

                 ┌──→ final_agent ──→ END
                 │
calculate_bill ──┤
                 │
                 └──→ food_agent

In [15]:
workflow.add_conditional_edges(
    "calculate_bill",
    check_budget,
    {
        "final_agent": "final_agent",
        "food_agent": "food_agent",
    }
)

In [16]:
workflow.add_edge("final_agent", END)

In [17]:
app = workflow.compile()

In [18]:
initial_state = {
    "user_request": "I want Egyptian food for 2 people with a budget of 300 EGP",

    "people": 2,
    "budget": 300,
    "preference": "traditional Egyptian",

    "recommended_food": None,
    "ingredients": [],

    "restaurant": None,
    "price_per_person": None,

    "total_cost": None,
    "remaining_budget": None,

    "budget_status": None,
    "final_response": None
}

In [19]:
result = app.invoke(initial_state)

print(result["final_response"])

Recommended food: Koshary
Restaurant: Koshary Place A
People: 2
Total cost: 100 EGP
Remaining budget: 200 EGP
